In [1]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import io
import hashlib
from pathlib import Path

c:\ITBA\MINICONDA\envs\TrabajoPractico_1\Lib\site-packages\albumentations\__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.7'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [2]:
import mlflow
import mlflow.pytorch

In [3]:
# Creamos el "experimento" en MLflow
mlflow.set_experiment("MLP_Clasificador_Imagenes")

<Experiment: artifact_location='file:///c:/ITBA/REDES%20NEURONALES/Tp1-Redes-Neuronales/mlruns/548689065550430374', creation_time=1779236188962, experiment_id='548689065550430374', last_update_time=1779236188962, lifecycle_stage='active', name='MLP_Clasificador_Imagenes', tags={}>

In [4]:
from torch.utils.tensorboard import SummaryWriter
import torchvision.utils as vutils

In [5]:
# Función para loguear una figura matplotlib en TensorBoard
def plot_to_tensorboard(fig, writer, tag, step):
    buf = io.BytesIO()
    fig.savefig(buf, format='png')
    buf.seek(0)
    image = Image.open(buf).convert("RGB")
    image = np.array(image)
    image = torch.tensor(image).permute(2, 0, 1) / 255.0
    writer.add_image(tag, image, global_step=step)
    plt.close(fig)

In [6]:
# Función para matriz de confusión y clasificación
global device         
global train_dataset

def log_classification_report(model, loader, writer, step, prefix="val"):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    fig_cm, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=train_dataset.label_encoder.classes_)
    disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
    ax.set_title(f'{prefix.title()} - Confusion Matrix')

    # Guardar localmente y subir a MLflow
    fig_path = f"confusion_matrix_{prefix}_epoch_{step}.png"
    fig_cm.savefig(fig_path)
    mlflow.log_artifact(fig_path)
    os.remove(fig_path)

    plot_to_tensorboard(fig_cm, writer, f"{prefix}/confusion_matrix", step)

    cls_report = classification_report(all_labels, all_preds, target_names= train_dataset.label_encoder.classes_)
    writer.add_text(f"{prefix}/classification_report", f"<pre>{cls_report}</pre>", step)

    # También loguear texto del reporte
    with open(f"classification_report_{prefix}_epoch_{step}.txt", "w") as f:
        f.write(cls_report)
    mlflow.log_artifact(f.name)
    os.remove(f.name)


In [7]:
# Crear directorio de logs
log_dir = "runs/mlp_experimento_1"
writer = SummaryWriter(log_dir=log_dir)


In [8]:
# Clase que le dice a PyTorch cómo leer nuestras imágenes, recorre las carpetas, asocia cada imagen con su clase, y aplica los transforms (resize, augmentations, normalización)

class CustomImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        self.image_paths = []
        self.labels = []

        class_names = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}

        for cls in class_names:
            cls_dir = os.path.join(root_dir, cls)
            for fname in os.listdir(cls_dir):
                if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                    self.image_paths.append(os.path.join(cls_dir, fname))
                    self.labels.append(cls)

        self.label_encoder = LabelEncoder()
        self.labels = self.label_encoder.fit_transform(self.labels)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = np.array(Image.open(self.image_paths[idx]).convert("RGB"))
        label = self.labels[idx]

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented["image"]

        return image, label

In [9]:
# TRANSFORMS DE TRAIN: augmentations para hacer el modelo más robusto. 

train_transform = A.Compose([
    A.Resize(64, 64), #Resize a 64x64
    A.HorizontalFlip(p=0.5), #Flip horizontal aleatorio
    A.RandomBrightnessContrast(p=0.2), #Variaciones de brillo/contraste
    A.Normalize(), #Normalización estándar
    ToTensorV2()
])


In [10]:
# TRANSFORMS DE VAL: sin augmentations, solo resize y normalizar (no queremos modificar las imágenes de validación)

val_test_transform = A.Compose([
    A.Resize(64, 64),
    A.Normalize(),
    ToTensorV2()
])

In [11]:
# 1. Rutas base originales
train_dir = Path("data/Split_smol/train")
val_dir = Path("data/Split_smol/val")

# 2. Leemos la carpeta de TRAIN completa con glob
all_train_paths = [p for p in train_dir.glob("**/*") if p.suffix.lower() in ['.jpg', '.jpeg', '.png']]


# RECONSTRUCCIÓN  DEL SPLIT (igual al EDA)

def get_class(x): return x.parent.name
valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

files_val = []
for x in val_dir.rglob('*'):
    if x.is_file() and x.suffix.lower() in valid_extensions:
        try:
            with Image.open(x) as img:
                files_val.append((x, get_class(x), img.size, img.mode))
        except Exception:
            pass

df_val_completo = pd.DataFrame(files_val, columns=["path", "class", "resolution", "mode"])

# REPRODUCIBILIDAD
df_val_completo = df_val_completo.sort_values(by="path").reset_index(drop=True)
np.random.seed(42)
df_test = df_val_completo.groupby('class', group_keys=False).apply(
    lambda x: x.sample(frac=0.5, random_state=42)
)


df_val_recortado = df_val_completo.drop(df_test.index).reset_index(drop=True)
df_test = df_test.reset_index(drop=True)


all_val_paths = [Path(p) for p in df_val_recortado['path'].tolist()]
all_test_paths = [Path(p) for p in df_test['path'].tolist()]


#INSPECCION DUPLICADOS
train_hashes = {}
for p in all_train_paths:
    with open(p, "rb") as f:
        file_hash = hashlib.md5(f.read()).hexdigest()
    train_hashes[file_hash] = p.name


# Filtrar TRAIN: Sacamos la foto negra y el duplicado interno
foto_negra_train = "ISIC_0031430.jpg"
duplicado_interno_train = "ISIC_0031039.jpg"
train_image_paths = [
    str(p) for p in all_train_paths
    if p.name != foto_negra_train and p.name != duplicado_interno_train
]

# Filtrar VAL NUEVO contra TRAIN
val_image_paths = []
count_leakage_val = 0
for p in all_val_paths:
    with open(p, "rb") as f:
        file_hash = hashlib.md5(f.read()).hexdigest()
    if file_hash in train_hashes:
        count_leakage_val += 1
    else:
        val_image_paths.append(str(p))

# Filtrar TEST contra TRAIN
test_image_paths = []
count_leakage_test = 0
for p in all_test_paths:
    with open(p, "rb") as f:
        file_hash = hashlib.md5(f.read()).hexdigest()
    if file_hash in train_hashes:
        count_leakage_test += 1
    else:
        test_image_paths.append(str(p))

C:\Users\Sofia\AppData\Local\Temp\ipykernel_3620\740683318.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_test = df_val_completo.groupby('class', group_keys=False).apply(


In [12]:
# DATALOADERS:  envuelven el dataset y lo sirven en batches
# shuffle=True en train para que no vea siempre el mismo orden

train_dataset = CustomImageDataset(train_dir, transform=train_transform)
val_dataset   = CustomImageDataset(val_dir, transform=val_test_transform)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size)

In [13]:
# MODELO : MLP simple. Aplana la imagen (64x64x3 = 12288 píxeles)

class MLPClassifier(nn.Module):
    def __init__(self, num_classes, input_size=64*64*3):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(), # Aplana la imagen (64x64x3 = 12288 píxeles)
            
            # --- CAPA 1 ---
            nn.Linear(input_size, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.25),           # <--- El tapón sagrado contra el overfitting
            
            # --- CAPA 2 ---
            nn.Linear(512, 128),        # <--- Conectamos 512 con 128
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0),              # <--- Flujo libre a la salida
            
            # --- CAPA DE SALIDA ---
            nn.Linear(128, num_classes) # <--- Conectamos 128 con tus clases fijas
        )

    def forward(self, x):
        return self.model(x)

In [14]:
# Loss: CrossEntropy (estándar para clasificación multiclase)
# Optimizer:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(set(train_dataset.labels))
model = MLPClassifier(num_classes=num_classes).to(device)


criterion = nn.CrossEntropyLoss()


optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)




In [15]:
# Entrenamiento y validación
# FUNCIÓN evaluate — corre el modelo sobre un loader
# calcula loss y accuracy, loguea métricas y primeras imágenes


def evaluate(model, loader, epoch=None, prefix="val"):
    model.eval()
    log_classification_report(model, loader, writer, step=epoch, prefix="val")
    correct, total, loss_sum = 0, 0, 0.0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for i, (images, labels) in enumerate(loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            loss_sum += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            # Loguear imágenes del primer batch
            if i == 0 and epoch is not None:
                img_grid = vutils.make_grid(images[:8].cpu(), normalize=True)
                writer.add_image(f"{prefix}/images", img_grid, global_step=epoch)

    acc = 100.0 * correct / total
    avg_loss = loss_sum / len(loader)

    if epoch is not None:
        writer.add_scalar(f"{prefix}/loss", avg_loss, epoch)
        writer.add_scalar(f"{prefix}/accuracy", acc, epoch)

    return avg_loss, acc

In [16]:
# LOOP DE ENTRENAMIENTO — 10 épocas
# por cada época: entrena, evalúa, loguea todo en MLflow
# al final guarda el modelo

n_epochs = 15
with mlflow.start_run():
    # Log hiperparámetros
    mlflow.log_params({
        "model": "MLPClassifier",
        "input_size": 64*64*3,
        "batch_size": batch_size,
        "lr": 0.0001,
        "epochs": n_epochs,
        "optimizer": "Adam",
        "loss_fn": "CrossEntropyLoss",
        "train_dir": train_dir,
        "val_dir": val_dir,
    })
    for epoch in range(n_epochs):
        model.train()
        running_loss = 0.0
        correct, total = 0, 0
    
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs}"):
            images, labels = images.to(device), labels.to(device)
    
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
        train_loss = running_loss / len(train_loader)
        train_acc = 100.0 * correct / total
        val_loss, val_acc = evaluate(model, val_loader, epoch=epoch, prefix="val")
    
        print(f"Epoch {epoch+1}:")
        print(f"  Train Loss: {train_loss:.4f}, Accuracy: {train_acc:.2f}%")
        print(f"  Val   Loss: {val_loss:.4f}, Accuracy: {val_acc:.2f}%")
    
        writer.add_scalar("train/loss", train_loss, epoch)
        writer.add_scalar("train/accuracy", train_acc, epoch)
    
        # Log en MLflow
        mlflow.log_metrics({
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc
        }, step=epoch)
        # Guardar modelo
    torch.save(model.state_dict(), "mlp_model.pth")
    print("Modelo guardado como 'mlp_model.pth'")
    mlflow.log_artifact("mlp_model.pth")
    mlflow.pytorch.log_model(model, artifact_path="pytorch_model")
    print("Modelo guardado como 'mlp_model.pth'")

Epoch 1/15: 100%|██████████| 22/22 [00:10<00:00,  2.04it/s]


Epoch 1:
  Train Loss: 1.8234, Accuracy: 36.49%
  Val   Loss: 1.6528, Accuracy: 46.11%


Epoch 2/15: 100%|██████████| 22/22 [00:10<00:00,  2.08it/s]


Epoch 2:
  Train Loss: 1.5780, Accuracy: 49.28%
  Val   Loss: 1.5811, Accuracy: 46.67%


Epoch 3/15: 100%|██████████| 22/22 [00:10<00:00,  2.04it/s]


...

Epoch 14:
  Train Loss: 0.9900, Accuracy: 71.70%
  Val   Loss: 1.1643, Accuracy: 59.44%


Epoch 15/15: 100%|██████████| 22/22 [00:09<00:00,  2.28it/s]


Epoch 15:
  Train Loss: 0.9581, Accuracy: 73.56%
  Val   Loss: 1.1305, Accuracy: 61.67%
Modelo guardado como 'mlp_model.pth'


2026/05/21 16:04:45 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Modelo guardado como 'mlp_model.pth'


In [17]:
!tensorboard --logdir=runs/mlp_experimento_1

^C
